# Cochleogram Generation — Deep Dive

This notebook explains **what cochleograms are**, **why we use them** for respiratory sound analysis,
and walks through **every step** of their generation from raw audio.

### Contents
1. Background: what is a cochleogram and how does it differ from a mel-spectrogram?
2. The ERB (Equivalent Rectangular Bandwidth) scale
3. Gammatone filter bank construction
4. Step-by-step generation with `pycochleagram`
5. Fallback path: librosa mel-spectrogram
6. Effect of key hyperparameters (`n_filters`, `low_lim`, `high_lim`, compression)
7. Saving cochleograms to disk (`.npy`)
8. Loading pre-saved cochleograms into the PyTorch Dataset

---
> **Reference:** Norman-Haignere & McDermott (2018), *pycochleagram* —  
> https://github.com/mcdermottLab/pycochleagram

## Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.signal
import librosa
import librosa.display
import torch
import torchaudio
from pathlib import Path

# Check pycochleagram availability
try:
    import cochleagram as cgram
    PYCOCHLEAGRAM = True
    print('pycochleagram  : AVAILABLE ✓')
except ImportError:
    PYCOCHLEAGRAM = False
    print('pycochleagram  : NOT FOUND — mel-spectrogram fallback will be used.')
    print('Install with:  pip install git+https://github.com/mcdermottLab/pycochleagram.git')

print(f'torch          : {torch.__version__}')
print(f'torchaudio     : {torchaudio.__version__}')
print(f'librosa        : {librosa.__version__}')

RAW_DIR   = Path('../data/raw')
PROC_DIR  = Path('../data/processed/cochleograms')
PROC_DIR.mkdir(parents=True, exist_ok=True)

SR           = 22050   # target sample rate
CLIP_SECONDS = 5.0
CLIP_SAMPLES = int(SR * CLIP_SECONDS)

---
## 1. What is a Cochleogram?

A **cochleogram** (also spelled *cochleagram*) is a time-frequency representation of a sound signal that
models how the **human cochlea** (inner ear) processes audio.

| Property | Mel-spectrogram | Cochleogram |
|---|---|---|
| Frequency scale | Mel (perceptual approximation) | ERB (Equivalent Rectangular Bandwidth) |
| Filter shape | Triangular / overlapping | Gammatone bandpass |
| Envelope extraction | STFT magnitude | Hilbert transform |
| Compression | Log | Power-law (x^0.3) |
| Biological fidelity | Moderate | High |

### Why does it matter for respiratory sounds?
Crackles and wheezes are **transient, wideband events** (crackles: 100–2000 Hz bursts;
wheezes: 100–1000 Hz sustained tones). The cochlea's gammatone filters have **better
time-frequency resolution trade-off** in this range compared to STFT-based representations,
which is why the reference paper reports improved classification performance over mel-spectrograms.

---
## 2. The ERB Scale

The ERB scale maps frequency to a perceptual spacing that matches the *bandwidth* of auditory filters
measured psychoacoustically (Glasberg & Moore, 1990):

$$\text{ERB}(f) = 24.7 \times (0.00437 f + 1)$$

The cumulative ERB number ("ERB-rate") is used to space filter center frequencies
so that each filter covers approximately **one critical band**.

In [ ]:
def erb_bandwidth(f_hz):
    """Glasberg & Moore (1990) ERB bandwidth at frequency f_hz."""
    return 24.7 * (0.00437 * f_hz + 1.0)

def hz_to_erb_rate(f_hz):
    """Convert Hz to ERB-rate (number of ERBs from 0 Hz)."""
    return 21.4 * np.log10(0.00437 * f_hz + 1.0)

def erb_rate_to_hz(erb_rate):
    """Inverse: ERB-rate → Hz."""
    return (10 ** (erb_rate / 21.4) - 1.0) / 0.00437

# Construct N=128 center frequencies between 50 Hz and 8000 Hz on the ERB scale
N_FILTERS = 128
LOW_LIM, HIGH_LIM = 50.0, 8000.0

erb_low  = hz_to_erb_rate(LOW_LIM)
erb_high = hz_to_erb_rate(HIGH_LIM)
erb_centers = np.linspace(erb_low, erb_high, N_FILTERS)
center_freqs_hz = erb_rate_to_hz(erb_centers)

# Compare ERB vs Mel vs Linear spacing
mel_centers = librosa.mel_frequencies(n_mels=N_FILTERS, fmin=LOW_LIM, fmax=HIGH_LIM)
lin_centers = np.linspace(LOW_LIM, HIGH_LIM, N_FILTERS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(range(N_FILTERS), center_freqs_hz, label='ERB', linewidth=2)
axes[0].plot(range(N_FILTERS), mel_centers,     label='Mel', linewidth=2, linestyle='--')
axes[0].plot(range(N_FILTERS), lin_centers,     label='Linear', linewidth=2, linestyle=':')
axes[0].set_xlabel('Filter index'); axes[0].set_ylabel('Center frequency (Hz)')
axes[0].set_title('Filter center frequencies: ERB vs Mel vs Linear')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

bw = erb_bandwidth(center_freqs_hz)
axes[1].fill_between(center_freqs_hz, center_freqs_hz - bw/2, center_freqs_hz + bw/2,
                     alpha=0.4, label='ERB filter coverage')
axes[1].plot(center_freqs_hz, bw, color='C1', label='ERB bandwidth')
axes[1].set_xscale('log'); axes[1].set_xlabel('Center frequency (Hz)')
axes[1].set_ylabel('Bandwidth (Hz)'); axes[1].set_title('ERB Bandwidth vs Frequency')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'ERB filter centers range: {center_freqs_hz[0]:.1f} Hz → {center_freqs_hz[-1]:.1f} Hz')
print(f'Lowest ERB bandwidth : {bw[0]:.1f} Hz  |  Highest: {bw[-1]:.1f} Hz')

---
## 3. Gammatone Filter Bank

Each cochlear filter is modelled as a **gammatone filter** — a sinusoidal carrier
multiplied by a gamma-function envelope:

$$g(t) = t^{n-1} e^{-2\pi b t} \cos(2\pi f_c t + \phi)$$

where:
- $f_c$ = center frequency
- $b$ = ERB-bandwidth-related decay (sharpness)
- $n = 4$ (order, matching human auditory filter bandwidth)

The impulse response below shows the characteristic **asymmetric, band-pass** shape.

In [ ]:
def gammatone_ir(fc, sr, duration=0.05, order=4):
    """Approximate gammatone impulse response for visualization."""
    t = np.linspace(0, duration, int(sr * duration))
    b = 1.019 * erb_bandwidth(fc)   # ERB decay rate
    envelope = t ** (order - 1) * np.exp(-2 * np.pi * b * t)
    carrier   = np.cos(2 * np.pi * fc * t)
    ir = envelope * carrier
    ir /= np.max(np.abs(ir) + 1e-10)  # normalize
    return t, ir

showcase_freqs = [100, 300, 800, 2000, 5000]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(showcase_freqs)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Time domain impulse responses
for fc, c in zip(showcase_freqs, colors):
    t, ir = gammatone_ir(fc, SR)
    axes[0].plot(t * 1000, ir, label=f'{fc} Hz', color=c, linewidth=1.2)
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('Gammatone Impulse Responses'); axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 30); axes[0].grid(True, alpha=0.3)

# Frequency responses
for fc, c in zip(showcase_freqs, colors):
    _, ir = gammatone_ir(fc, SR, duration=0.1)
    freqs = np.fft.rfftfreq(len(ir), d=1/SR)
    mag   = np.abs(np.fft.rfft(ir))
    mag   = 20 * np.log10(mag / mag.max() + 1e-10)
    axes[1].plot(freqs, mag, label=f'{fc} Hz', color=c, linewidth=1.2)
axes[1].set_xscale('log'); axes[1].set_xlim(LOW_LIM, HIGH_LIM)
axes[1].set_ylim(-60, 5); axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude (dB)'); axes[1].set_title('Gammatone Frequency Responses')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Step-by-Step Cochleogram Generation

The pipeline has **5 stages**:

```
Audio signal
    │
    ▼  [1] Gammatone filter bank  (ERB-spaced, FFT convolution)
N sub-band signals
    │
    ▼  [2] Hilbert envelope extraction  (analytical signal → |z(t)|)
N envelope signals
    │
    ▼  [3] Downsampling  (optional temporal compression)
N downsampled envelopes
    │
    ▼  [4] Non-linear compression  (x^0.3 power law)
N compressed envelopes
    │
    ▼  [5] Stack → 2D cochleogram  (n_filters × n_time_frames)
```

We first load a real audio file, then walk through each stage.

In [ ]:
# -----------------------------------------------------------------------
# Load an audio clip — use a real file if available, otherwise synthesize
# -----------------------------------------------------------------------
wav_files = sorted(RAW_DIR.glob('*.wav'))

if wav_files:
    wav_path = wav_files[0]
    waveform, orig_sr = torchaudio.load(wav_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if orig_sr != SR:
        waveform = torchaudio.transforms.Resample(orig_sr, SR)(waveform)
    audio = waveform.squeeze().numpy()[:CLIP_SAMPLES]
    if len(audio) < CLIP_SAMPLES:
        audio = np.pad(audio, (0, CLIP_SAMPLES - len(audio)))
    source_label = f'Real file: {wav_path.name}'
else:
    # Synthesize a signal that resembles a crackle burst + wheeze tone
    t = np.linspace(0, CLIP_SECONDS, CLIP_SAMPLES)
    # Wheeze: 400 Hz sustained tone with AM
    wheeze = 0.4 * np.sin(2 * np.pi * 400 * t) * (0.5 + 0.5 * np.sin(2 * np.pi * 3 * t))
    # Crackle: short Gaussian bursts at random times
    rng = np.random.default_rng(42)
    crackle = np.zeros_like(t)
    for burst_time in rng.uniform(0.5, 4.5, size=12):
        idx = int(burst_time * SR)
        burst_len = int(0.005 * SR)
        if idx + burst_len < len(crackle):
            burst = rng.normal(0, 1, burst_len) * np.hanning(burst_len)
            crackle[idx:idx + burst_len] += burst
    audio = wheeze + crackle * 0.8
    audio /= np.max(np.abs(audio) + 1e-8)
    source_label = 'Synthesized signal (wheeze + crackle)'

time_axis = np.linspace(0, CLIP_SECONDS, len(audio))
print(f'Source    : {source_label}')
print(f'Duration  : {len(audio)/SR:.2f}s  |  Samples: {len(audio)}  |  SR: {SR} Hz')

plt.figure(figsize=(13, 2))
plt.plot(time_axis, audio, linewidth=0.5, color='steelblue')
plt.title(f'Waveform — {source_label}')
plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
plt.tight_layout(); plt.show()

### Stage 1 — Gammatone filtering

We show the output of 5 representative filters at different frequency bands.

In [ ]:
# Manual gammatone filtering via overlap-add FFT convolution (visualization only)
def apply_gammatone_filter(signal, fc, sr, order=4, duration=0.05):
    """Filter `signal` with a gammatone centered at `fc` Hz."""
    _, ir = gammatone_ir(fc, sr, duration=duration, order=order)
    return scipy.signal.fftconvolve(signal, ir, mode='same')

demo_freqs = [100, 400, 1000, 3000, 7000]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(demo_freqs)))

fig, axes = plt.subplots(len(demo_freqs), 1, figsize=(13, 8), sharex=True)
for ax, fc, c in zip(axes, demo_freqs, colors):
    filtered = apply_gammatone_filter(audio, fc, SR)
    ax.plot(time_axis, filtered, color=c, linewidth=0.6)
    ax.set_ylabel(f'{fc} Hz', rotation=0, labelpad=45, va='center', fontsize=9)
    ax.set_yticks([])
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Stage 1: Gammatone Sub-band Signals', fontsize=13)
plt.tight_layout()
plt.show()

### Stage 2 — Hilbert envelope extraction

The **instantaneous amplitude** (envelope) of each sub-band is extracted via the analytic signal:

$$\text{envelope}(t) = |\mathcal{H}\{g(t)\}| = \sqrt{g(t)^2 + \hat{g}(t)^2}$$

where $\hat{g}(t)$ is the Hilbert transform of the filtered signal.

In [ ]:
fig, axes = plt.subplots(len(demo_freqs), 1, figsize=(13, 8), sharex=True)
for ax, fc, c in zip(axes, demo_freqs, colors):
    filtered  = apply_gammatone_filter(audio, fc, SR)
    analytic  = scipy.signal.hilbert(filtered)
    envelope  = np.abs(analytic)
    ax.plot(time_axis, filtered,  color=c, linewidth=0.4, alpha=0.4, label='filtered')
    ax.plot(time_axis, envelope,  color=c, linewidth=1.2, label='envelope')
    ax.set_ylabel(f'{fc} Hz', rotation=0, labelpad=45, va='center', fontsize=9)
    ax.set_yticks([])
    ax.grid(True, alpha=0.2)

axes[0].legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Stage 2: Sub-band Envelope (Hilbert)', fontsize=13)
plt.tight_layout()
plt.show()

### Stage 3 — Stack envelopes → raw cochleogram matrix

In [ ]:
# Build a lightweight cochleogram using N_FILTERS filters (no pycochleagram required)
# This illustrates the concept; pycochleagram uses FFT convolution for speed.

N_DEMO = 64   # use 64 filters to keep runtime reasonable in the notebook

erb_low  = hz_to_erb_rate(LOW_LIM)
erb_high = hz_to_erb_rate(HIGH_LIM)
erb_c    = np.linspace(erb_low, erb_high, N_DEMO)
fc_demo  = erb_rate_to_hz(erb_c)

cg_raw = np.zeros((N_DEMO, len(audio)), dtype=np.float32)
for i, fc in enumerate(fc_demo):
    filtered       = apply_gammatone_filter(audio, fc, SR)
    cg_raw[i, :]   = np.abs(scipy.signal.hilbert(filtered))

print(f'Raw cochleogram shape: {cg_raw.shape}  (n_filters × n_samples)')

plt.figure(figsize=(13, 5))
plt.imshow(cg_raw, aspect='auto', origin='lower',
           extent=[0, CLIP_SECONDS, LOW_LIM, HIGH_LIM], cmap='magma')
plt.colorbar(label='Envelope amplitude')
plt.yscale('log'); plt.ylabel('Frequency (Hz)'); plt.xlabel('Time (s)')
plt.title('Stage 3: Raw Stacked Envelopes (pre-compression)')
plt.tight_layout(); plt.show()

### Stage 4 — Power-law compression

The cochlea compresses its dynamic range via a **power-law non-linearity** ($x^{0.3}$)
rather than the log used in mel-spectrograms.  
This more closely matches measured auditory nerve responses.

In [ ]:
# Compare compression functions
x = np.linspace(0, 1, 500)
plt.figure(figsize=(7, 4))
plt.plot(x, x ** 0.3,                              label='Power law  x^0.3',    linewidth=2)
plt.plot(x, np.log1p(x * 100) / np.log1p(100),    label='Log  log(1+100x)',    linewidth=2, linestyle='--')
plt.plot(x, x,                                      label='Linear  (no compression)', linewidth=1.5, linestyle=':')
plt.xlabel('Input amplitude'); plt.ylabel('Compressed output')
plt.title('Compression Non-linearities'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Apply to our raw cochleogram
cg_compressed = cg_raw ** 0.3

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(cg_raw,        aspect='auto', origin='lower', cmap='magma',
               extent=[0, CLIP_SECONDS, 0, N_DEMO])
axes[0].set_title('Before compression'); axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('ERB channel')
axes[1].imshow(cg_compressed, aspect='auto', origin='lower', cmap='magma',
               extent=[0, CLIP_SECONDS, 0, N_DEMO])
axes[1].set_title('After x^0.3 compression'); axes[1].set_xlabel('Time (s)')
plt.suptitle('Stage 4: Power-law Compression', fontsize=13)
plt.tight_layout(); plt.show()

### Stage 5 — Normalize and resize to ViT input shape

In [ ]:
# Normalize to [0, 1]
cg_norm = (cg_compressed - cg_compressed.min()) / (cg_compressed.max() - cg_compressed.min() + 1e-8)

# Resize to 128 × 128 using bilinear interpolation (PyTorch)
TARGET = 128
cg_tensor = torch.from_numpy(cg_norm).unsqueeze(0).unsqueeze(0)   # (1, 1, H, W)
cg_resized = torch.nn.functional.interpolate(
    cg_tensor, size=(TARGET, TARGET), mode='bilinear', align_corners=False
).squeeze().numpy()   # (128, 128)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cg_norm,    aspect='auto', origin='lower', cmap='magma')
axes[0].set_title(f'Normalized  ({N_DEMO} × {len(audio)} → compressed to view)')
axes[0].set_xlabel('Time frames'); axes[0].set_ylabel('ERB channel')

axes[1].imshow(cg_resized, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title(f'Resized to {TARGET}×{TARGET} — ViT input')
axes[1].set_xlabel('Time frames'); axes[1].set_ylabel('ERB channel')

plt.suptitle('Stage 5: Normalization & Resize', fontsize=13)
plt.tight_layout(); plt.show()

print('Final tensor shape (1, H, W):', torch.from_numpy(cg_resized).unsqueeze(0).shape)

---
## 5. Using the `CochleogramTransform` class

All the steps above are encapsulated in `CochleogramTransform`.
When `pycochleagram` is available it uses the full pipeline (FFT-convolving all filters at once, much faster).
Otherwise it falls back to the librosa mel path.

In [ ]:
from cochleogram_vit.preprocessing.cochleogram import CochleogramTransform

transform = CochleogramTransform(
    sr=SR,
    n_filters=128,
    low_lim=50.0,
    high_lim=8000.0,
    sample_factor=1,
    downsample=None,
    output_size=128,
)

clip_tensor = torch.from_numpy(audio).unsqueeze(0)  # (1, n_samples)
cg_out = transform(clip_tensor)                      # (1, 128, 128)

print('Output shape :', cg_out.shape)
print('Value range  : [{:.4f}, {:.4f}]'.format(cg_out.min().item(), cg_out.max().item()))

plt.figure(figsize=(6, 5))
plt.imshow(cg_out.squeeze().numpy(), origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='Normalized amplitude')
plt.title('CochleogramTransform output (128×128)')
plt.xlabel('Time frames'); plt.ylabel('ERB channel')
plt.tight_layout(); plt.show()

---
## 6. Effect of Key Hyperparameters

We sweep over the most important knobs to show their effect on the cochleogram.

In [ ]:
# ---- (A) Number of filters ----
filter_counts = [32, 64, 128, 256]

fig, axes = plt.subplots(1, len(filter_counts), figsize=(14, 4))
for ax, n in zip(axes, filter_counts):
    t = CochleogramTransform(sr=SR, n_filters=n, output_size=128)
    img = t(clip_tensor).squeeze().numpy()
    ax.imshow(img, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
    ax.set_title(f'n_filters={n}')
    ax.set_xlabel('Time'); ax.set_ylabel('ERB ch.')
fig.suptitle('(A) Effect of n_filters', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ---- (B) Frequency range ----
freq_ranges = [(50, 4000), (50, 8000), (100, 8000), (200, 11000)]

fig, axes = plt.subplots(1, len(freq_ranges), figsize=(14, 4))
for ax, (lo, hi) in zip(axes, freq_ranges):
    t = CochleogramTransform(sr=SR, n_filters=128, low_lim=lo, high_lim=hi, output_size=128)
    img = t(clip_tensor).squeeze().numpy()
    ax.imshow(img, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
    ax.set_title(f'{lo}–{hi} Hz')
    ax.set_xlabel('Time')
fig.suptitle('(B) Effect of frequency range', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ---- (C) Compression exponent (applied manually for illustration) ----
exponents = [0.1, 0.3, 0.5, 1.0]

fig, axes = plt.subplots(1, len(exponents), figsize=(14, 4))
for ax, exp in zip(axes, exponents):
    # Build raw cg, apply different exponent
    raw = cg_raw.copy()
    raw_norm = (raw ** exp)
    raw_norm = (raw_norm - raw_norm.min()) / (raw_norm.max() - raw_norm.min() + 1e-8)
    # Resize
    t_raw = torch.from_numpy(raw_norm).unsqueeze(0).unsqueeze(0).float()
    img = torch.nn.functional.interpolate(t_raw, (128, 128), mode='bilinear', align_corners=False)
    img = img.squeeze().numpy()
    ax.imshow(img, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
    ax.set_title(f'exp={exp}')
    ax.set_xlabel('Time')
fig.suptitle('(C) Effect of compression exponent  (default=0.3)', fontsize=13)
plt.tight_layout(); plt.show()

---
## 7. Cochleogram vs Mel-Spectrogram — Direct Comparison

Key visual differences to highlight in your presentation:
- **Finer low-frequency resolution** in cochleogram (more ERB channels below 500 Hz)
- **Smoother temporal envelopes** (Hilbert vs STFT magnitude)
- **More biological fidelity** — directly models inner-hair-cell activation patterns

In [ ]:
# Mel-spectrogram
mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=128, fmin=50, fmax=8000)
log_mel = librosa.power_to_db(mel, ref=np.max)
log_mel_norm = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)

# Cochleogram (via our transform)
cg_final = transform(clip_tensor).squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(log_mel_norm, origin='lower', aspect='auto', cmap='magma',
               extent=[0, CLIP_SECONDS, 50, 8000])
axes[0].set_yscale('log')
axes[0].set_title('Mel-Spectrogram (128 bins, log scale)')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Frequency (Hz)')

axes[1].imshow(cg_final, origin='lower', aspect='auto', cmap='magma',
               extent=[0, CLIP_SECONDS, 50, 8000])
axes[1].set_yscale('log')
axes[1].set_title('Cochleogram (128 ERB filters, x^0.3 compression)')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Frequency (Hz)')

plt.suptitle('Mel-Spectrogram vs Cochleogram — same audio clip', fontsize=13)
plt.tight_layout(); plt.show()

---
## 8. Saving Cochleograms to Disk

For training we pre-generate all cochleograms once and save them as `.npy` files
to avoid repeating the computation every epoch.
The full batch version is in `scripts/preprocess.py`.

Here we show the single-file version:

In [ ]:
import numpy as np
from pathlib import Path

save_path = PROC_DIR / 'demo_cochleogram.npy'

# cg_final is shape (128, 128), dtype float32
np.save(save_path, cg_final)
print(f'Saved to : {save_path}')
print(f'Shape    : {cg_final.shape}  |  dtype: {cg_final.dtype}')
print(f'File size: {save_path.stat().st_size / 1024:.1f} KB')

# Reload and verify
reloaded = np.load(save_path)
assert np.allclose(reloaded, cg_final), 'Mismatch after reload!'
print('Reload check: OK')

---
## 9. Loading Pre-Saved Cochleograms into the PyTorch Dataset

Once `scripts/preprocess.py` has run, you can load from `.npy` files directly
(much faster than recomputing on every epoch).

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


class PrecomputedCochleogramDataset(Dataset):
    """
    Loads pre-generated cochleograms from .npy files.
    Requires data/processed/metadata.csv produced by scripts/preprocess.py.
    """
    def __init__(self, metadata_csv: str, augment: bool = False):
        self.df = pd.read_csv(metadata_csv)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cg = np.load(row['npy_path'])                        # (H, W)  float32
        tensor = torch.from_numpy(cg).unsqueeze(0)           # (1, H, W)

        if self.augment:
            tensor = self._augment(tensor)

        return tensor, int(row['label'])

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        """Lightweight augmentations suitable for cochleograms."""
        # Random horizontal flip (time reversal — physically valid for envelopes)
        if torch.rand(1).item() > 0.5:
            x = x.flip(-1)
        # Additive Gaussian noise
        x = x + torch.randn_like(x) * 0.01
        x = x.clamp(0.0, 1.0)
        return x


# Demo: create a tiny synthetic metadata CSV and test the dataset
demo_meta = pd.DataFrame({
    'npy_path': [str(PROC_DIR / 'demo_cochleogram.npy')],
    'label':    [1],
    'label_name': ['crackle'],
})
demo_meta_path = PROC_DIR / 'demo_metadata.csv'
demo_meta.to_csv(demo_meta_path, index=False)

ds = PrecomputedCochleogramDataset(str(demo_meta_path), augment=True)
loader = DataLoader(ds, batch_size=1)
x_batch, y_batch = next(iter(loader))
print('Batch shape  :', x_batch.shape)    # (1, 1, 128, 128)
print('Label        :', y_batch.item())    # 1 (crackle)

---
## Summary

| Step | Operation | Key Parameter |
|------|-----------|---------------|
| 1 | Gammatone filter bank | `n_filters=128`, `low_lim=50`, `high_lim=8000` |
| 2 | Hilbert envelope extraction | — |
| 3 | (Optional) temporal downsampling | `downsample=None` |
| 4 | Power-law compression | exponent = 0.3 |
| 5 | Normalize to [0, 1] | — |
| 6 | Resize to square | `output_size=128` |

**To reproduce the full preprocessing run:**
```bash
# Put ICBHI .wav and .txt files in data/raw/, then:
python scripts/preprocess.py --config configs/default.yaml
```

The output is `data/processed/cochleograms/*.npy` + `data/processed/metadata.csv`.